In [ ]:
print("Supreme")

Supreme


In [ ]:
# https://www.kaggle.com/datasets/praveengovi/emotions-dataset-for-nlp/data

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/

In [ ]:
!kaggle datasets download -d praveengovi/emotions-dataset-for-nlp

Dataset URL: https://www.kaggle.com/datasets/praveengovi/emotions-dataset-for-nlp
License(s): CC-BY-SA-4.0
100% 721k/721k [00:00<00:00, 131MB/s]



In [ ]:
import zipfile
zip_ref = zipfile.ZipFile('/content/emotions-dataset-for-nlp.zip', 'r')
zip_ref.extractall('/content')
zip_ref.close()

In [ ]:
# Import the pandas library for data manipulation
import pandas as pd

# Load the training dataset from 'train.txt'
# The data is separated by semicolons and has 'text' and 'emotion' columns
train_df = pd.read_csv('/content/train.txt', names=['text', 'emotion'], sep=';')

# Load the testing dataset from 'test.txt'
# The data is separated by semicolons and has 'text' and 'emotion' columns
test_df = pd.read_csv('/content/test.txt', names=['text', 'emotion'], sep=';')

# Load the validation dataset from 'val.txt'
# The data is separated by semicolons and has 'text' and 'emotion' columns
val_df = pd.read_csv('/content/val.txt', names=['text', 'emotion'], sep=';')

# Display the first few rows of the training DataFrame to check the data
print("Train DataFrame head:")
print(train_df.head())

# Display the first few rows of the testing DataFrame
print("\nTest DataFrame head:")
print(test_df.head())

# Display the first few rows of the validation DataFrame
print("\nValidation DataFrame head:")
print(val_df.head())

Train DataFrame head:
                                                text  emotion
0                            i didnt feel humiliated  sadness
1  i can go from feeling so hopeless to so damned...  sadness
2   im grabbing a minute to post i feel greedy wrong    anger
3  i am ever feeling nostalgic about the fireplac...     love
4                               i am feeling grouchy    anger

Test DataFrame head:
                                                text  emotion
0  im feeling rather rotten so im not very ambiti...  sadness
1          im updating my blog because i feel shitty  sadness
2  i never make her separate from me because i do...  sadness
3  i left with my bouquet of red and yellow tulip...      joy
4    i was feeling a little vain when i did this one  sadness

Validation DataFrame head:
                                                text  emotion
0  im feeling quite sad and sorry for myself but ...  sadness
1  i feel like i am still looking at a blank canv...  sadnes

In [ ]:
# Importing necessary libraries for text processing
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re

# Download NLTK data if not already downloaded
# 'stopwords' are common words that are often removed from text
# 'wordnet' is a lexical database used for lemmatization
nltk.download('stopwords')
nltk.download('wordnet')

# Initialize the WordNet Lemmatizer
lemmatizer = WordNetLemmatizer()

# Get the English stopwords set
stop_words = set(stopwords.words('english'))

# Define a function to clean the text
def clean_text(text):
    # Convert text to lowercase
    text = text.lower()
    # Remove special characters, numbers, and punctuation
    # Keep only alphabetic characters and replace others with a space
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    # Tokenize the text (split into words)
    words = text.split()
    # Remove stopwords and perform lemmatization
    # Lemmatization reduces words to their base or root form (e.g., 'running' -> 'run')
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    # Join the cleaned words back into a single string
    return ' '.join(words)

# Apply the cleaning function to the 'text' column of each DataFrame
print("Applying text cleaning to training data...")
train_df['cleaned_text'] = train_df['text'].apply(clean_text)
print("Applying text cleaning to testing data...")
test_df['cleaned_text'] = test_df['text'].apply(clean_text)
print("Applying text cleaning to validation data...")
val_df['cleaned_text'] = val_df['text'].apply(clean_text)

# Display the head of the DataFrames with the new 'cleaned_text' column
print("\nTrain DataFrame after cleaning:")
print(train_df[['text', 'cleaned_text', 'emotion']].head())

print("\nTest DataFrame after cleaning:")
print(test_df[['text', 'cleaned_text', 'emotion']].head())

print("\nValidation DataFrame after cleaning:")
print(val_df[['text', 'cleaned_text', 'emotion']].head())

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


Applying text cleaning to training data...
Applying text cleaning to testing data...
Applying text cleaning to validation data...

Train DataFrame after cleaning:
                                                text  \
0                            i didnt feel humiliated   
1  i can go from feeling so hopeless to so damned...   
2   im grabbing a minute to post i feel greedy wrong   
3  i am ever feeling nostalgic about the fireplac...   
4                               i am feeling grouchy   

                                        cleaned_text  emotion  
0                              didnt feel humiliated  sadness  
1  go feeling hopeless damned hopeful around some...  sadness  
2          im grabbing minute post feel greedy wrong    anger  
3  ever feeling nostalgic fireplace know still pr...     love  
4                                    feeling grouchy    anger  

Test DataFrame after cleaning:
                                                text  \
0  im feeling rather rotten 

In [ ]:
# Importing necessary libraries for feature engineering
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# --- Bag of Words (CountVectorizer) ---
print("\n--- Performing Bag of Words (CountVectorizer) ---")
# Initialize CountVectorizer. It converts a collection of text documents to a matrix of token counts.
# Each row represents a document, and each column represents a word in the vocabulary.
count_vectorizer = CountVectorizer(max_features=5000) # Limiting to top 5000 features for manageable size

# Fit the vectorizer on the training data and transform it
# This step learns the vocabulary from the training data
X_train_bow = count_vectorizer.fit_transform(train_df['cleaned_text'])

# Transform the test and validation data using the SAME fitted vectorizer
# This ensures consistency in the vocabulary used across datasets
X_test_bow = count_vectorizer.transform(test_df['cleaned_text'])
X_val_bow = count_vectorizer.transform(val_df['cleaned_text'])

# Display the shape of the resulting matrices
print(f"Shape of X_train_bow: {X_train_bow.shape}") # (number of training samples, number of unique words)
print(f"Shape of X_test_bow: {X_test_bow.shape}")
print(f"Shape of X_val_bow: {X_val_bow.shape}")




--- Performing Bag of Words (CountVectorizer) ---
Shape of X_train_bow: (16000, 5000)
Shape of X_test_bow: (2000, 5000)
Shape of X_val_bow: (2000, 5000)


In [ ]:
# --- TF-IDF (TfidfVectorizer) ---
print("\n--- Performing TF-IDF (TfidfVectorizer) ---")
# Initialize TfidfVectorizer. It transforms text into a matrix of TF-IDF features.
# TF-IDF (Term Frequency-Inverse Document Frequency) reflects how important a word is to a document in a corpus.
tfidf_vectorizer = TfidfVectorizer(max_features=5000) # Limiting to top 5000 features for consistency

# Fit the vectorizer on the training data and transform it
# This step calculates IDF values based on the training data
X_train_tfidf = tfidf_vectorizer.fit_transform(train_df['cleaned_text'])

# Transform the test and validation data using the SAME fitted vectorizer
X_test_tfidf = tfidf_vectorizer.transform(test_df['cleaned_text'])
X_val_tfidf = tfidf_vectorizer.transform(val_df['cleaned_text'])

# Display the shape of the resulting matrices
print(f"Shape of X_train_tfidf: {X_train_tfidf.shape}") # (number of training samples, number of unique words)
print(f"Shape of X_test_tfidf: {X_test_tfidf.shape}")
print(f"Shape of X_val_tfidf: {X_val_tfidf.shape}")


--- Performing TF-IDF (TfidfVectorizer) ---
Shape of X_train_tfidf: (16000, 5000)
Shape of X_test_tfidf: (2000, 5000)
Shape of X_val_tfidf: (2000, 5000)


In [ ]:
# --- Word2Vec Embedding ---
print("\n--- Performing Word2Vec Embedding ---")

!pip install gensim
# Importing necessary libraries for Word2Vec
from gensim.models import Word2Vec
import numpy as np

# Tokenize the cleaned text for Word2Vec input
# Word2Vec model expects a list of lists, where each inner list is a document (list of words)
tokenized_train_text = [text.split() for text in train_df['cleaned_text']]
tokenized_test_text = [text.split() for text in test_df['cleaned_text']]
tokenized_val_text = [text.split() for text in val_df['cleaned_text']]

# Train the Word2Vec model on the training data
# vector_size: Dimensionality of the word vectors
# window: Maximum distance between the current and predicted word within a sentence
# min_count: Ignores all words with total frequency lower than this
# sg: Training algorithm (1 for skip-gram, 0 for CBOW)
print("Training Word2Vec model...")
word2vec_model = Word2Vec(sentences=tokenized_train_text, vector_size=100, window=5, min_count=5, workers=4, sg=0)
print("Word2Vec model training complete.")

# Function to create document embeddings by averaging word vectors
def document_vector(model, doc):
    # Filter out words that are not in the model's vocabulary
    doc = [word for word in doc if word in model.wv.index_to_key]
    if len(doc) == 0:
        return np.zeros(model.vector_size) # Return a zero vector if no words are found
    # Average the word vectors for the words in the document
    return np.mean(model.wv[doc], axis=0)

# Create Word2Vec embeddings for the training, test, and validation datasets
print("Generating Word2Vec embeddings for training data...")
X_train_word2vec = np.array([document_vector(word2vec_model, doc) for doc in tokenized_train_text])
print("Generating Word2Vec embeddings for testing data...")
X_test_word2vec = np.array([document_vector(word2vec_model, doc) for doc in tokenized_test_text])
print("Generating Word2Vec embeddings for validation data...")
X_val_word2vec = np.array([document_vector(word2vec_model, doc) for doc in tokenized_val_text])

# Display the shape of the resulting matrices
print(f"Shape of X_train_word2vec: {X_train_word2vec.shape}") # (number of training samples, vector_size)
print(f"Shape of X_test_word2vec: {X_test_word2vec.shape}")
print(f"Shape of X_val_word2vec: {X_val_word2vec.shape}")


--- Performing Word2Vec Embedding ---
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 53.3 MB/s eta 0:00:00
Training Word2Vec model...
Word2Vec model training complete.
Generating Word2Vec embeddings for training data...
Generating Word2Vec embeddings for testing data...
Generating Word2Vec embeddings for validation data...
Shape of X_train_word2vec: (16000, 100)
Shape of X_test_word2vec: (2000, 100)
Shape of X_val_word2vec: (2000, 100)


In [ ]:
# Importing necessary libraries for machine learning models and evaluation
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np



In [ ]:
# --- Prepare Target Variable (Emotion Labels) ---
print("\n--- Preparing Target Variable ---")
# Initialize LabelEncoder to convert categorical emotion labels into numerical labels
label_encoder = LabelEncoder()

# Fit and transform the 'emotion' column of the training data
# This step learns all unique emotion labels and assigns a unique integer to each
y_train = label_encoder.fit_transform(train_df['emotion'])

# Transform the 'emotion' column of the test and validation data using the fitted encoder
# This ensures that the same mapping is applied consistently across all datasets
y_test = label_encoder.transform(test_df['emotion'])
y_val = label_encoder.transform(val_df['emotion'])

# Display the mapping of labels to integers and the shape of the encoded targets
print("Emotion labels mapping:", dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")
print(f"Shape of y_val: {y_val.shape}")




--- Preparing Target Variable ---
Emotion labels mapping: {'anger': np.int64(0), 'fear': np.int64(1), 'joy': np.int64(2), 'love': np.int64(3), 'sadness': np.int64(4), 'surprise': np.int64(5)}
Shape of y_train: (16000,)
Shape of y_test: (2000,)
Shape of y_val: (2000,)


In [ ]:
# --- Function to Train and Evaluate Models ---
def train_and_evaluate_model(model, X_train, y_train, X_test, y_test, feature_name, model_name):
    print(f"\nTraining {model_name} with {feature_name} features...")
    try:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
        recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
        f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

        print(f"--- {model_name} Results using {feature_name} ---")
        print(f"Accuracy: {accuracy:.4f}")
        print(f"Precision: {precision:.4f}")
        print(f"Recall: {recall:.4f}")
        print(f"F1-Score: {f1:.4f}")
    except Exception as e:
        print(f"Error training {model_name} with {feature_name}: {e}")


In [ ]:

# --- Initialize Models ---
# Logistic Regression: A linear model used for classification
logistic_regression_model = LogisticRegression(max_iter=1000, solver='liblinear', random_state=42)

# Support Vector Classifier (SVC): Powerful for high-dimensional data, can be slow on large datasets
svm_model = SVC(kernel='linear', random_state=42) # Using 'linear' kernel for efficiency with text data

# Multinomial Naive Bayes: Suitable for discrete features like word counts (Bag of Words, TF-IDF)
# Not suitable for dense, continuous data like Word2Vec embeddings
multinomial_nb_model = MultinomialNB(alpha=1.0) # alpha is a smoothing parameter (Laplace smoothing)

# Random Forest Classifier: An ensemble method building multiple decision trees
random_forest_model = RandomForestClassifier(n_estimators=100, random_state=42)

# Gradient Boosting Classifier: Another powerful ensemble technique
gradient_boosting_model = GradientBoostingClassifier(n_estimators=100, random_state=42)

# K-Nearest Neighbors (KNN): Non-parametric, classification based on closest training examples
knn_model = KNeighborsClassifier(n_neighbors=5)

# Decision Tree Classifier: Simple tree-based model
decision_tree_model = DecisionTreeClassifier(random_state=42)



In [ ]:
# List of models to evaluate
models = {
    "Logistic Regression": logistic_regression_model,
    "SVC": svm_model,
    "Multinomial Naive Bayes": multinomial_nb_model,
    "Random Forest": random_forest_model,
    "Gradient Boosting": gradient_boosting_model,
    "K-Nearest Neighbors": knn_model,
    "Decision Tree": decision_tree_model
}


In [ ]:

# --- Evaluate Models with Bag of Words features ---
print("\n======================================================")
print("--- Evaluating Models with Bag of Words Features ---")
print("======================================================")
for name, model in models.items():
    train_and_evaluate_model(model, X_train_bow, y_train, X_test_bow, y_test, "Bag of Words", name)



--- Evaluating Models with Bag of Words Features ---

Training Logistic Regression with Bag of Words features...
--- Logistic Regression Results using Bag of Words ---
Accuracy: 0.8965
Precision: 0.8954
Recall: 0.8965
F1-Score: 0.8958

Training SVC with Bag of Words features...
--- SVC Results using Bag of Words ---
Accuracy: 0.8795
Precision: 0.8801
Recall: 0.8795
F1-Score: 0.8794

Training Multinomial Naive Bayes with Bag of Words features...
--- Multinomial Naive Bayes Results using Bag of Words ---
Accuracy: 0.8430
Precision: 0.8408
Recall: 0.8430
F1-Score: 0.8341

Training Random Forest with Bag of Words features...
--- Random Forest Results using Bag of Words ---
Accuracy: 0.8810
Precision: 0.8821
Recall: 0.8810
F1-Score: 0.8814

Training Gradient Boosting with Bag of Words features...
--- Gradient Boosting Results using Bag of Words ---
Accuracy: 0.8435
Precision: 0.8572
Recall: 0.8435
F1-Score: 0.8440

Training K-Nearest Neighbors with Bag of Words features...
--- K-Nearest Ne

In [ ]:

# --- Evaluate Models with TF-IDF features ---
print("\n======================================================")
print("--- Evaluating Models with TF-IDF Features ---")
print("======================================================")
for name, model in models.items():
    train_and_evaluate_model(model, X_train_tfidf, y_train, X_test_tfidf, y_test, "TF-IDF", name)




--- Evaluating Models with TF-IDF Features ---

Training Logistic Regression with TF-IDF features...
--- Logistic Regression Results using TF-IDF ---
Accuracy: 0.8720
Precision: 0.8746
Recall: 0.8720
F1-Score: 0.8669

Training SVC with TF-IDF features...
--- SVC Results using TF-IDF ---
Accuracy: 0.8865
Precision: 0.8853
Recall: 0.8865
F1-Score: 0.8851

Training Multinomial Naive Bayes with TF-IDF features...
--- Multinomial Naive Bayes Results using TF-IDF ---
Accuracy: 0.7550
Precision: 0.7702
Recall: 0.7550
F1-Score: 0.7174

Training Random Forest with TF-IDF features...
--- Random Forest Results using TF-IDF ---
Accuracy: 0.8915
Precision: 0.8908
Recall: 0.8915
F1-Score: 0.8907

Training Gradient Boosting with TF-IDF features...
--- Gradient Boosting Results using TF-IDF ---
Accuracy: 0.8420
Precision: 0.8561
Recall: 0.8420
F1-Score: 0.8425

Training K-Nearest Neighbors with TF-IDF features...
--- K-Nearest Neighbors Results using TF-IDF ---
Accuracy: 0.7685
Precision: 0.7907
Reca

In [ ]:
# --- Evaluate Models with Word2Vec features ---
print("\n======================================================")
print("--- Evaluating Models with Word2Vec Features ---")
print("======================================================")
# MultinomialNB is skipped for Word2Vec because it expects count data, not dense embeddings
for name, model in models.items():
    if name == "Multinomial Naive Bayes":
        print(f"\nSkipping {name} with Word2Vec features as it is not suitable for dense, continuous data.")
        continue
    train_and_evaluate_model(model, X_train_word2vec, y_train, X_test_word2vec, y_test, "Word2Vec", name)



--- Evaluating Models with Word2Vec Features ---

Training Logistic Regression with Word2Vec features...
--- Logistic Regression Results using Word2Vec ---
Accuracy: 0.3620
Precision: 0.2274
Recall: 0.3620
F1-Score: 0.2542

Training SVC with Word2Vec features...
--- SVC Results using Word2Vec ---
Accuracy: 0.3495
Precision: 0.2186
Recall: 0.3495
F1-Score: 0.2072

Skipping Multinomial Naive Bayes with Word2Vec features as it is not suitable for dense, continuous data.

Training Random Forest with Word2Vec features...
--- Random Forest Results using Word2Vec ---
Accuracy: 0.3775
Precision: 0.3444
Recall: 0.3775
F1-Score: 0.3120

Training Gradient Boosting with Word2Vec features...
--- Gradient Boosting Results using Word2Vec ---
Accuracy: 0.3760
Precision: 0.3784
Recall: 0.3760
F1-Score: 0.3008

Training K-Nearest Neighbors with Word2Vec features...
--- K-Nearest Neighbors Results using Word2Vec ---
Accuracy: 0.2925
Precision: 0.2803
Recall: 0.2925
F1-Score: 0.2813

Training Decision Tr

In [ ]:
# Importing necessary libraries for hyperparameter tuning
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform

print("\n--- Performing Hyperparameter Tuning ---")

# --- 1. Hyperparameter Tuning for Logistic Regression with TF-IDF ---
print("\n--- Tuning Logistic Regression with TF-IDF ---")
# Define the parameter distribution for Logistic Regression
# C: Inverse of regularization strength; smaller values specify stronger regularization.
# solver: Algorithm to use in the optimization problem.
# `loguniform` is used for `C` as it samples efficiently across a wide range.
param_dist_lr = {
    'C': loguniform(0.1, 100), # Regularization strength
    'solver': ['liblinear', 'saga'] # Solvers that work well with L1/L2 regularization
}

# Initialize Logistic Regression model
lr = LogisticRegression(random_state=42, max_iter=1000)

# Initialize RandomizedSearchCV
# n_iter: Number of parameter settings that are sampled.
# cv: Number of folds for cross-validation.
# scoring: Metric to evaluate the performance of the cross-validated model.
# n_jobs: Number of jobs to run in parallel (-1 means using all processors).
random_search_lr = RandomizedSearchCV(
    estimator=lr,
    param_distributions=param_dist_lr,
    n_iter=20, # Reduced for faster execution
    cv=5,
    scoring='f1_weighted',
    verbose=1,
    n_jobs=-1,
    random_state=42
)

# Fit RandomizedSearchCV to the training data (TF-IDF features)
random_search_lr.fit(X_train_tfidf, y_train)

# Get the best estimator and its parameters
best_lr_model = random_search_lr.best_estimator_
best_lr_params = random_search_lr.best_params_

print(f"\nBest parameters for Logistic Regression (TF-IDF): {best_lr_params}")

# Evaluate the best Logistic Regression model on the test set
print("Evaluating best Logistic Regression model...")
train_and_evaluate_model(best_lr_model, X_train_tfidf, y_train, X_test_tfidf, y_test, "TF-IDF (Tuned)", "Logistic Regression")




--- Performing Hyperparameter Tuning ---

--- Tuning Logistic Regression with TF-IDF ---
Fitting 5 folds for each of 20 candidates, totalling 100 fits

Best parameters for Logistic Regression (TF-IDF): {'C': np.float64(6.251373574521748), 'solver': 'liblinear'}
Evaluating best Logistic Regression model...

Training Logistic Regression with TF-IDF (Tuned) features...
--- Logistic Regression Results using TF-IDF (Tuned) ---
Accuracy: 0.8925
Precision: 0.8915
Recall: 0.8925
F1-Score: 0.8912


In [ ]:
# --- 2. Hyperparameter Tuning for Random Forest with TF-IDF ---
print("\n--- Tuning Random Forest with TF-IDF ---")
# Define the parameter distribution for Random Forest
# n_estimators: Number of trees in the forest.
# max_depth: Maximum number of levels in a tree.
# min_samples_split: Minimum number of samples required to split an internal node.
# min_samples_leaf: Minimum number of samples required to be at a leaf node.
param_dist_rf = {
    'n_estimators': [100, 200, 300], # Number of trees
    'max_depth': [10, 20, 30, None], # Maximum depth of the tree (None means unlimited)
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Initialize Random Forest model
rf = RandomForestClassifier(random_state=42)

# Initialize RandomizedSearchCV
random_search_rf = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist_rf,
    n_iter=10, # Reduced for faster execution
    cv=3, # Reduced for faster execution
    scoring='f1_weighted',
    verbose=1,
    n_jobs=-1,
    random_state=42
)

# Fit RandomizedSearchCV to the training data (TF-IDF features)
random_search_rf.fit(X_train_tfidf, y_train)

# Get the best estimator and its parameters
best_rf_model = random_search_rf.best_estimator_
best_rf_params = random_search_rf.best_params_

print(f"\nBest parameters for Random Forest (TF-IDF): {best_rf_params}")

# Evaluate the best Random Forest model on the test set
print("Evaluating best Random Forest model...")
train_and_evaluate_model(best_rf_model, X_train_tfidf, y_train, X_test_tfidf, y_test, "TF-IDF (Tuned)", "Random Forest")



--- Tuning Random Forest with TF-IDF ---
Fitting 3 folds for each of 10 candidates, totalling 30 fits

Best parameters for Random Forest (TF-IDF): {'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_depth': None}
Evaluating best Random Forest model...

Training Random Forest with TF-IDF (Tuned) features...
--- Random Forest Results using TF-IDF (Tuned) ---
Accuracy: 0.8855
Precision: 0.8848
Recall: 0.8855
F1-Score: 0.8846


In [ ]:

# --- 3. Hyperparameter Tuning for SVC with TF-IDF ---
print("\n--- Tuning SVC with TF-IDF ---")
param_dist_svc = {
    'C': loguniform(0.1, 100),
    'kernel': ['linear'] # Linear kernel is often effective for text classification with TF-IDF
}

svc = SVC(random_state=42)
random_search_svc = RandomizedSearchCV(
    estimator=svc,
    param_distributions=param_dist_svc,
    n_iter=10,
    cv=3,
    scoring='f1_weighted',
    verbose=1,
    n_jobs=-1,
    random_state=42
)
random_search_svc.fit(X_train_tfidf, y_train)
best_svc_model = random_search_svc.best_estimator_
best_svc_params = random_search_svc.best_params_
print(f"\nBest parameters for SVC (TF-IDF): {best_svc_params}")
print("Evaluating best SVC model...")
train_and_evaluate_model(best_svc_model, X_train_tfidf, y_train, X_test_tfidf, y_test, "TF-IDF (Tuned)", "SVC")




--- Tuning SVC with TF-IDF ---
Fitting 3 folds for each of 10 candidates, totalling 30 fits

Best parameters for SVC (TF-IDF): {'C': np.float64(1.3292918943162166), 'kernel': 'linear'}
Evaluating best SVC model...

Training SVC with TF-IDF (Tuned) features...
--- SVC Results using TF-IDF (Tuned) ---
Accuracy: 0.8870
Precision: 0.8861
Recall: 0.8870
F1-Score: 0.8863


In [ ]:
# --- 4. Hyperparameter Tuning for Multinomial Naive Bayes with TF-IDF ---
print("\n--- Tuning Multinomial Naive Bayes with TF-IDF ---")
param_dist_mnb = {
    'alpha': loguniform(0.01, 10) # Smoothing parameter for Naive Bayes
}

mnb_tfidf = MultinomialNB()
random_search_mnb_tfidf = RandomizedSearchCV(
    estimator=mnb_tfidf,
    param_distributions=param_dist_mnb,
    n_iter=10,
    cv=3,
    scoring='f1_weighted',
    verbose=1,
    n_jobs=-1,
    random_state=42
)
random_search_mnb_tfidf.fit(X_train_tfidf, y_train)
best_mnb_tfidf_model = random_search_mnb_tfidf.best_estimator_
best_mnb_tfidf_params = random_search_mnb_tfidf.best_params_
print(f"\nBest parameters for Multinomial Naive Bayes (TF-IDF): {best_mnb_tfidf_params}")
print("Evaluating best Multinomial Naive Bayes (TF-IDF) model...")
train_and_evaluate_model(best_mnb_tfidf_model, X_train_tfidf, y_train, X_test_tfidf, y_test, "TF-IDF (Tuned)", "Multinomial Naive Bayes")



--- Tuning Multinomial Naive Bayes with TF-IDF ---
Fitting 3 folds for each of 10 candidates, totalling 30 fits

Best parameters for Multinomial Naive Bayes (TF-IDF): {'alpha': np.float64(0.13292918943162169)}
Evaluating best Multinomial Naive Bayes (TF-IDF) model...

Training Multinomial Naive Bayes with TF-IDF (Tuned) features...
--- Multinomial Naive Bayes Results using TF-IDF (Tuned) ---
Accuracy: 0.7895
Precision: 0.8054
Recall: 0.7895
F1-Score: 0.7727


In [ ]:

# --- 4b. Hyperparameter Tuning for Multinomial Naive Bayes with Bag of Words ---
print("\n--- Tuning Multinomial Naive Bayes with Bag of Words ---")
mnb_bow = MultinomialNB()
random_search_mnb_bow = RandomizedSearchCV(
    estimator=mnb_bow,
    param_distributions=param_dist_mnb, # Using the same alpha distribution
    n_iter=10,
    cv=3,
    scoring='f1_weighted',
    verbose=1,
    n_jobs=-1,
    random_state=42
)
random_search_mnb_bow.fit(X_train_bow, y_train)
best_mnb_bow_model = random_search_mnb_bow.best_estimator_
best_mnb_bow_params = random_search_mnb_bow.best_params_
print(f"\nBest parameters for Multinomial Naive Bayes (Bag of Words): {best_mnb_bow_params}")
print("Evaluating best Multinomial Naive Bayes (Bag of Words) model...")
train_and_evaluate_model(best_mnb_bow_model, X_train_bow, y_train, X_test_bow, y_test, "Bag of Words (Tuned)", "Multinomial Naive Bayes")



--- Tuning Multinomial Naive Bayes with Bag of Words ---
Fitting 3 folds for each of 10 candidates, totalling 30 fits

Best parameters for Multinomial Naive Bayes (Bag of Words): {'alpha': np.float64(0.6251373574521749)}
Evaluating best Multinomial Naive Bayes (Bag of Words) model...

Training Multinomial Naive Bayes with Bag of Words (Tuned) features...
--- Multinomial Naive Bayes Results using Bag of Words (Tuned) ---
Accuracy: 0.8450
Precision: 0.8416
Recall: 0.8450
F1-Score: 0.8405


In [ ]:
# --- 6. Hyperparameter Tuning for K-Nearest Neighbors with TF-IDF ---
print("\n--- Tuning K-Nearest Neighbors with TF-IDF ---")
param_dist_knn = {
    'n_neighbors': [3, 5, 7, 9],
    'weights': ['uniform', 'distance'] # Weight function used in prediction
}

knn = KNeighborsClassifier()
random_search_knn = RandomizedSearchCV(
    estimator=knn,
    param_distributions=param_dist_knn,
    n_iter=10,
    cv=3,
    scoring='f1_weighted',
    verbose=1,
    n_jobs=-1,
    random_state=42
)
random_search_knn.fit(X_train_tfidf, y_train)
best_knn_model = random_search_knn.best_estimator_
best_knn_params = random_search_knn.best_params_
print(f"\nBest parameters for K-Nearest Neighbors (TF-IDF): {best_knn_params}")
print("Evaluating best K-Nearest Neighbors model...")
train_and_evaluate_model(best_knn_model, X_train_tfidf, y_train, X_test_tfidf, y_test, "TF-IDF (Tuned)", "K-Nearest Neighbors")




--- Tuning K-Nearest Neighbors with TF-IDF ---
Fitting 3 folds for each of 8 candidates, totalling 24 fits


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 8 is smaller than n_iter=10. Running 8 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(



Best parameters for K-Nearest Neighbors (TF-IDF): {'weights': 'distance', 'n_neighbors': 9}
Evaluating best K-Nearest Neighbors model...

Training K-Nearest Neighbors with TF-IDF (Tuned) features...
--- K-Nearest Neighbors Results using TF-IDF (Tuned) ---
Accuracy: 0.8085
Precision: 0.8128
Recall: 0.8085
F1-Score: 0.8059


In [ ]:
# --- 7. Hyperparameter Tuning for Decision Tree with TF-IDF ---
print("\n--- Tuning Decision Tree with TF-IDF ---")
param_dist_dt = {
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

dt = DecisionTreeClassifier(random_state=42)
random_search_dt = RandomizedSearchCV(
    estimator=dt,
    param_distributions=param_dist_dt,
    n_iter=10,
    cv=3,
    scoring='f1_weighted',
    verbose=1,
    n_jobs=-1,
    random_state=42
)
random_search_dt.fit(X_train_tfidf, y_train)
best_dt_model = random_search_dt.best_estimator_
best_dt_params = random_search_dt.best_params_
print(f"\nBest parameters for Decision Tree (TF-IDF): {best_dt_params}")
print("Evaluating best Decision Tree model...")
train_and_evaluate_model(best_dt_model, X_train_tfidf, y_train, X_test_tfidf, y_test, "TF-IDF (Tuned)", "Decision Tree")



--- Tuning Decision Tree with TF-IDF ---
Fitting 3 folds for each of 10 candidates, totalling 30 fits

Best parameters for Decision Tree (TF-IDF): {'min_samples_split': 10, 'min_samples_leaf': 4, 'max_depth': None}
Evaluating best Decision Tree model...

Training Decision Tree with TF-IDF (Tuned) features...
--- Decision Tree Results using TF-IDF (Tuned) ---
Accuracy: 0.8650
Precision: 0.8665
Recall: 0.8650
F1-Score: 0.8655


In [ ]:
# --- Function to store and evaluate models ---
# This function will now return the metrics, so we can collect them for comparison.
def get_model_performance(model, X_train, y_train, X_test, y_test, feature_name, model_name):
    """Trains a model and returns its performance metrics."""
    # Ensure model is refitted if it's not already, particularly important for tuned models
    # if the model object itself is not already the best_estimator_.
    # For this setup, we assume best_lr_model, best_rf_model, etc., are already fitted,
    # but we'll refit for consistency in collecting results.
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

    return {
        'Model': model_name,
        'Features': feature_name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1
    }

# List to store all model results
all_results = []

print("\n--- Re-collecting Initial Model Performances ---")
# --- Evaluate Initial Models with Bag of Words features ---
for name, model in models.items():
    results = get_model_performance(model, X_train_bow, y_train, X_test_bow, y_test, "Bag of Words", name)
    all_results.append(results)

# --- Evaluate Initial Models with TF-IDF features ---
for name, model in models.items():
    results = get_model_performance(model, X_train_tfidf, y_train, X_test_tfidf, y_test, "TF-IDF", name)
    all_results.append(results)

# --- Evaluate Initial Models with Word2Vec features ---
for name, model in models.items():
    if name == "Multinomial Naive Bayes":
        # Skip Multinomial Naive Bayes for Word2Vec as it's not suitable
        continue
    results = get_model_performance(model, X_train_word2vec, y_train, X_test_word2vec, y_test, "Word2Vec", name)
    all_results.append(results)

print("\n--- Re-collecting Tuned Model Performances ---")
# Add Tuned Model Results

# Logistic Regression (TF-IDF)
all_results.append(get_model_performance(best_lr_model, X_train_tfidf, y_train, X_test_tfidf, y_test, "TF-IDF (Tuned)", "Logistic Regression"))

# Random Forest (TF-IDF)
all_results.append(get_model_performance(best_rf_model, X_train_tfidf, y_train, X_test_tfidf, y_test, "TF-IDF (Tuned)", "Random Forest"))

# SVC (TF-IDF)
all_results.append(get_model_performance(best_svc_model, X_train_tfidf, y_train, X_test_tfidf, y_test, "TF-IDF (Tuned)", "SVC"))

# Multinomial Naive Bayes (TF-IDF)
all_results.append(get_model_performance(best_mnb_tfidf_model, X_train_tfidf, y_train, X_test_tfidf, y_test, "TF-IDF (Tuned)", "Multinomial Naive Bayes"))

# Multinomial Naive Bayes (Bag of Words)
all_results.append(get_model_performance(best_mnb_bow_model, X_train_bow, y_train, X_test_bow, y_test, "Bag of Words (Tuned)", "Multinomial Naive Bayes"))

# Gradient Boosting (TF-IDF)
# Check if best_gb_model is defined (i.e., if tuning cell was executed)
if 'best_gb_model' in locals():
    all_results.append(get_model_performance(best_gb_model, X_train_tfidf, y_train, X_test_tfidf, y_test, "TF-IDF (Tuned)", "Gradient Boosting"))
else:
    print("Warning: Tuned Gradient Boosting model (best_gb_model) not found. Using untuned version for comparison.")
    all_results.append(get_model_performance(models["Gradient Boosting"], X_train_tfidf, y_train, X_test_tfidf, y_test, "TF-IDF (Untuned Fallback)", "Gradient Boosting"))

# K-Nearest Neighbors (TF-IDF)
# Check if best_knn_model is defined (i.e., if tuning cell was executed)
if 'best_knn_model' in locals():
    all_results.append(get_model_performance(best_knn_model, X_train_tfidf, y_train, X_test_tfidf, y_test, "TF-IDF (Tuned)", "K-Nearest Neighbors"))
else:
    print("Warning: Tuned K-Nearest Neighbors model (best_knn_model) not found. Using untuned version for comparison.")
    all_results.append(get_model_performance(models["K-Nearest Neighbors"], X_train_tfidf, y_train, X_test_tfidf, y_test, "TF-IDF (Untuned Fallback)", "K-Nearest Neighbors"))

# Decision Tree (TF-IDF)
# Check if best_dt_model is defined (i.e., if tuning cell was executed)
if 'best_dt_model' in locals():
    all_results.append(get_model_performance(best_dt_model, X_train_tfidf, y_train, X_test_tfidf, y_test, "TF-IDF (Tuned)", "Decision Tree"))
else:
    print("Warning: Tuned Decision Tree model (best_dt_model) not found. Using untuned version for comparison.")
    all_results.append(get_model_performance(models["Decision Tree"], X_train_tfidf, y_train, X_test_tfidf, y_test, "TF-IDF (Untuned Fallback)", "Decision Tree"))


# Create a DataFrame from the results
results_df = pd.DataFrame(all_results)

# Sort by F1-Score for better readability
results_df_sorted = results_df.sort_values(by='F1-Score', ascending=False).reset_index(drop=True)

print("\n--- Model Performance Comparison Table ---")
print(results_df_sorted.to_markdown(index=False))

# Suggest the best model
best_model_row = results_df_sorted.iloc[0]
print(f"\n--- Best Performing Model ---")
print(f"Based on F1-Score, the best performing model is: {best_model_row['Model']} using {best_model_row['Features']} features.")
print(f"It achieved an F1-Score of {best_model_row['F1-Score']:.4f}, Accuracy of {best_model_row['Accuracy']:.4f}, Precision of {best_model_row['Precision']:.4f}, and Recall of {best_model_row['Recall']:.4f}.")


--- Re-collecting Initial Model Performances ---

--- Re-collecting Tuned Model Performances ---

--- Model Performance Comparison Table ---
| Model                   | Features                  |   Accuracy |   Precision |   Recall |   F1-Score |
|:------------------------|:--------------------------|-----------:|------------:|---------:|-----------:|
| Logistic Regression     | Bag of Words              |     0.8965 |    0.895412 |   0.8965 |   0.895796 |
| Logistic Regression     | TF-IDF (Tuned)            |     0.8925 |    0.891452 |   0.8925 |   0.891209 |
| Random Forest           | TF-IDF                    |     0.8915 |    0.890797 |   0.8915 |   0.890662 |
| SVC                     | TF-IDF (Tuned)            |     0.887  |    0.886135 |   0.887  |   0.886323 |
| SVC                     | TF-IDF                    |     0.8865 |    0.885255 |   0.8865 |   0.885074 |
| Random Forest           | TF-IDF (Tuned)            |     0.8855 |    0.88478  |   0.8855 |   0.884586 |
| 